# 08. Engagement 민감도 분석 + 긍정률 변화 분석

**입력**
- `data/review_individual.csv` — 개별 리뷰 (playtime, voted_up 포함)
- `data/discount_history.csv` — 할인 이벤트

**분석 3가지**
1. **Engagement 민감도 분석** — 플레이타임 필터 3기준(전체 / 2시간+ / 5시간+)별 Engagement 반응률·유지율
2. **긍정률 변화 (Sentiment Shift)** — 할인 전·중·후 구간별 voted_up 비율 변화
3. **장르별 플레이타임 분포** — 장르 분류의 데이터 기반 근거

**출력**: `figures/chart8~11.png`

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings('ignore')

def root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent]:
        if (c / 'data').exists() and (c / 'figures').exists():
            return c
    return cwd

PROJECT_ROOT = root()
DATA_DIR     = PROJECT_ROOT / 'data'
FIGURE_DIR   = PROJECT_ROOT / 'figures'
FIGURE_DIR.mkdir(exist_ok=True)

rv   = pd.read_csv(DATA_DIR / 'review_individual.csv')
disc = pd.read_csv(DATA_DIR / 'discount_history.csv')

disc['discount_start'] = pd.to_datetime(disc['discount_start'])
disc['discount_end']   = pd.to_datetime(disc['discount_end'])
rv['date'] = pd.to_datetime(rv['timestamp_created'], unit='s', utc=True).dt.tz_localize(None).dt.normalize()

print(f'리뷰: {len(rv):,}개 / 게임: {rv["app_id"].nunique()}개')
print(f'할인 이벤트: {len(disc)}개 / 게임: {disc["appid"].nunique()}개')

In [ ]:
# 폰트 설정
def kfont():
    for fp in [PROJECT_ROOT / 'fonts' / 'NanumGothic.ttc',
               PROJECT_ROOT / 'fonts' / 'NanumGothic.ttf']:
        if fp.exists():
            font_manager.fontManager.addfont(str(fp))
            return font_manager.FontProperties(fname=str(fp)).get_name()
    for fn in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'DejaVu Sans']:
        if fn in {f.name for f in font_manager.fontManager.ttflist}:
            return fn
    return 'DejaVu Sans'

FONT_NAME = kfont()
plt.rcParams.update({'font.family': FONT_NAME, 'axes.unicode_minus': False, 'figure.dpi': 120})

GENRE_ORDER  = ['RPG', 'Adventure', 'Strategy/Simulation', 'Casual/Lightweight', 'Action']
GENRE_COLORS = {
    'RPG':                  '#4C72B0',
    'Adventure':            '#DD8452',
    'Strategy/Simulation':  '#55A868',
    'Casual/Lightweight':   '#C44E52',
    'Action':               '#8172B2',
}
DPI = 300

PRE_DAYS  = 30
POST_DAYS = 14
MIN_PRE_DAYS  = 14
MIN_POST_DAYS = 7

print(f'폰트: {FONT_NAME}')

# 게임별 리뷰 시계열 사전 (app_id → date-indexed series)
rv_by_app = {appid: grp.set_index('date')['review_id']
             for appid, grp in rv.groupby('app_id')}
print('게임별 리뷰 시계열 준비 완료')

## 분석 1 — Engagement 민감도 분석

플레이타임 필터 3가지 기준으로 동일한 Engagement 반응률·유지율을 계산하여,
필터에 따라 결론이 바뀌는지 확인한다.

| 기준 | 포함 대상 |
|------|----------|
| A: 전체 | 필터 없음 |
| B: 2시간+ | playtime_at_review_min ≥ 120 |
| C: 5시간+ | playtime_at_review_min ≥ 300 |

In [ ]:
def window_avg(series, start, end):
    mask = (series.index >= start) & (series.index < end)
    sub  = series[mask]
    if len(sub) == 0:
        return np.nan
    return sub.sum() / (end - start).days


def clip95(s):
    lo, hi = np.percentile(s.dropna(), [5, 95])
    return s.clip(lo, hi)


def compute_engagement(rv_filtered, disc):
    """개별 리뷰 → 일별 집계 → 이벤트별 Engagement 반응률·유지율 계산."""
    # 일별 집계
    daily = (rv_filtered.groupby(['app_id', 'date'])['review_id']
             .count().reset_index()
             .rename(columns={'review_id': 'cnt', 'app_id': 'appid'}))
    daily['date'] = pd.to_datetime(daily['date'])

    series_map = {appid: grp.set_index('date')['cnt']
                  for appid, grp in daily.groupby('appid')}

    rows = []
    for _, ev in disc.iterrows():
        appid = int(ev['appid'])
        if appid not in series_map:
            continue
        s     = series_map[appid]
        start = ev['discount_start']
        end   = ev['discount_end']
        pre_s = start - pd.Timedelta(days=PRE_DAYS)
        post_e = end + pd.Timedelta(days=POST_DAYS)

        if (start - s.index.min()).days < MIN_PRE_DAYS:
            continue
        if (s.index.max() - end).days < MIN_POST_DAYS:
            continue

        b = window_avg(s, pre_s, start)
        d = window_avg(s, start, end)
        a = window_avg(s, end, post_e)

        if pd.isna(b) or b == 0:
            continue

        rows.append({
            'appid':          appid,
            'name':           ev['name'],
            'genre_category': ev['genre_category'],
            'reaction_rate':  (d - b) / b if not pd.isna(d) else np.nan,
            'sustained_rate': (a - b) / b if not pd.isna(a) else np.nan,
        })

    result = pd.DataFrame(rows)
    if len(result):
        result['reaction_rate']  = clip95(result['reaction_rate'])
        result['sustained_rate'] = clip95(result['sustained_rate'])
    return result


print('함수 정의 완료')

In [ ]:
filters = {
    'A: 전체':   rv,
    'B: 2시간+': rv[rv['playtime_at_review_min'] >= 120],
    'C: 5시간+': rv[rv['playtime_at_review_min'] >= 300],
}

results = {}
for label, rv_f in filters.items():
    df = compute_engagement(rv_f, disc)
    results[label] = df
    print(f'{label}: 유효 이벤트 {len(df)}개')

# 장르별 중앙값 집계
summary = {}
for label, df in results.items():
    summary[label] = (
        df.groupby('genre_category')[['reaction_rate', 'sustained_rate']]
        .median()
        .reindex(GENRE_ORDER)
        .round(3)
    )

print()
for label, s in summary.items():
    print(f'=== {label} ===')
    print(s.to_string())
    print()

In [ ]:
filter_labels  = list(summary.keys())
filter_colors  = ['#4472C4', '#ED7D31', '#A9D18E']
x = np.arange(len(GENRE_ORDER))
w = 0.25

for metric, metric_label, fname in [
    ('reaction_rate',  'Engagement 반응률 중앙값',  'chart8_sensitivity_response.png'),
    ('sustained_rate', 'Engagement 유지율 중앙값', 'chart9_sensitivity_retention.png'),
]:
    fig, ax = plt.subplots(figsize=(12, 6))
    for i, (label, color) in enumerate(zip(filter_labels, filter_colors)):
        vals = summary[label][metric].values
        bars = ax.bar(x + (i - 1) * w, vals, w, label=label,
                      color=color, edgecolor='black', alpha=0.85)
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                va  = 'bottom' if v >= 0 else 'top'
                off = 0.008 if v >= 0 else -0.008
                ax.text(bar.get_x() + bar.get_width() / 2, v + off,
                        f'{v:+.3f}', ha='center', va=va, fontsize=8)

    ax.axhline(0, color='gray', linestyle=':', linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(GENRE_ORDER)
    ax.set_ylabel(metric_label)
    ax.set_title(f'플레이타임 필터별 {metric_label} (장르 × 기준)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / fname, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f'저장 완료 {fname}')

# 결과 표
print()
print('=== Engagement 반응률 민감도 분석 결과 ===')
tbl_r = pd.DataFrame({lbl: summary[lbl]['reaction_rate']  for lbl in filter_labels})
tbl_s = pd.DataFrame({lbl: summary[lbl]['sustained_rate'] for lbl in filter_labels})
print('[ 반응률 ]')
print(tbl_r.round(3).to_string())
print()
print('[ 유지율 ]')
print(tbl_s.round(3).to_string())

## 분석 2 — 긍정률 변화 (Sentiment Shift)

할인 이벤트별로 직전 30일 / 할인 기간 / 종료 후 14일 구간의 `voted_up` 비율을 비교한다.

- 최소 표본 조건: 각 구간별 리뷰 **10개 이상**인 이벤트만 사용
- 표본 부족 이벤트는 별도 집계 후 장르 단위 보조 분석으로 제시

In [ ]:
MIN_REVIEWS = 10

rv_by_appid = {appid: grp for appid, grp in rv.groupby('app_id')}

sent_rows = []
skipped   = 0

for _, ev in disc.iterrows():
    appid = int(ev['appid'])
    if appid not in rv_by_appid:
        skipped += 1
        continue

    g      = rv_by_appid[appid]
    start  = ev['discount_start']
    end    = ev['discount_end']
    pre_s  = start - pd.Timedelta(days=PRE_DAYS)
    post_e = end   + pd.Timedelta(days=POST_DAYS)

    before = g[(g['date'] >= pre_s)  & (g['date'] < start)]
    during = g[(g['date'] >= start)  & (g['date'] < end)]
    after  = g[(g['date'] >= end)    & (g['date'] < post_e)]

    if len(before) < MIN_REVIEWS or len(during) < MIN_REVIEWS:
        skipped += 1
        continue

    sent_rows.append({
        'appid':            appid,
        'name':             ev['name'],
        'genre_category':   ev['genre_category'],
        'before_positive':  before['voted_up'].mean(),
        'during_positive':  during['voted_up'].mean(),
        'after_positive':   after['voted_up'].mean() if len(after) >= MIN_REVIEWS else np.nan,
        'n_before':         len(before),
        'n_during':         len(during),
        'n_after':          len(after),
    })

sent_df = pd.DataFrame(sent_rows)
sent_df['sentiment_shift'] = sent_df['during_positive'] - sent_df['before_positive']

total_events = len(disc)
print(f'총 이벤트 {total_events}건 중 {len(sent_df)}건 분석 가능 (표본 부족 제외: {skipped}건)')
print()

# 장르별 집계
genre_sent = (
    sent_df.groupby('genre_category')
    .agg(
        직전30일=('before_positive', 'mean'),
        할인기간=('during_positive', 'mean'),
        종료후14일=('after_positive', 'mean'),
        변화량=('sentiment_shift', 'mean'),
        이벤트수=('appid', 'count'),
    )
    .reindex(GENRE_ORDER)
    .round(3)
)
print('=== 장르별 긍정률 변화 ===')
print(genre_sent.to_string())

In [ ]:
periods    = ['직전30일', '할인기간', '종료후14일']
period_clr = ['#5B9BD5', '#ED7D31', '#A9D18E']
x = np.arange(len(GENRE_ORDER))
w = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (period, color) in enumerate(zip(periods, period_clr)):
    vals = genre_sent[period].values
    bars = ax.bar(x + (i - 1) * w, vals, w, label=period,
                  color=color, edgecolor='black', alpha=0.85)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.004,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(GENRE_ORDER)
ax.set_ylabel('긍정률 (voted_up 비율)')
ax.set_title('장르별 할인 전·중·후 긍정률 변화')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.text(0.98, 0.02, f'이벤트 기준 {len(sent_df)}건 (구간별 리뷰 {MIN_REVIEWS}개+ 조건)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart10_sentiment.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart10_sentiment.png')

print()
print('=== 할인 중 긍정률 변화량 (during - before) ===')
shift = genre_sent[['변화량', '이벤트수']].copy()
shift['방향'] = shift['변화량'].apply(lambda x: '↑ 상승' if x > 0.01 else ('↓ 하락' if x < -0.01 else '→ 유지'))
print(shift.to_string())

## 분석 3 — 장르별 플레이타임 분포

장르 분류가 단순 태그가 아니라 **실제 소비 행동 차이**를 반영한다는 데이터 기반 근거.

- `playtime_at_review_min` 기준 (리뷰 작성 시점 플레이타임)
- playtime = 0인 리뷰 제외
- y축 로그 스케일

In [ ]:
pt = rv[rv['playtime_at_review_min'] > 0].copy()
pt['playtime_hours'] = pt['playtime_at_review_min'] / 60

# 기초 통계
stats = (
    pt.groupby('genre')['playtime_hours']
    .agg(['median', 'mean',
          ('p25', lambda x: x.quantile(0.25)),
          ('p75', lambda x: x.quantile(0.75)),
          ('p95', lambda x: x.quantile(0.95))])
    .reindex(GENRE_ORDER)
    .round(1)
)
print('=== 장르별 플레이타임(시간) 분포 ===')
print(stats.to_string())
print()

# 박스플롯
data_by_genre = [
    pt[pt['genre'] == g]['playtime_hours'].values
    for g in GENRE_ORDER
]

fig, ax = plt.subplots(figsize=(11, 6))
bp = ax.boxplot(
    data_by_genre,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(color='black', linewidth=2),
)
for patch, genre in zip(bp['boxes'], GENRE_ORDER):
    patch.set_facecolor(GENRE_COLORS[genre])
    patch.set_alpha(0.75)

ax.set_yscale('log')
ax.set_xticks(range(1, len(GENRE_ORDER) + 1))
ax.set_xticklabels(GENRE_ORDER)
ax.set_ylabel('리뷰 작성 시점 플레이타임 (시간, 로그 스케일)')
ax.set_title('장르별 플레이타임 분포 (playtime = 0 제외, 이상치 미표시)')
ax.grid(axis='y', alpha=0.3)

# 중앙값 레이블
for i, genre in enumerate(GENRE_ORDER, 1):
    med = stats.loc[genre, 'median']
    ax.text(i, med * 1.15, f'중앙값\n{med:.1f}h',
            ha='center', va='bottom', fontsize=8)

ax.text(0.98, 0.02, f'리뷰 {len(pt):,}개 (playtime>0)',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=8, color='gray')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'chart11_playtime_by_genre.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('저장 완료 chart11_playtime_by_genre.png')